# Ítem 2 · Clasificación con ResNet50

Clasificación binaria: **0 = sin paneles, 1 = con paneles**. Ejecutar en orden en Colab con GPU T4.
Este notebook cubre la arquitectura ResNet50; la consigna completa exige además otra arquitectura.

| Experimento | Inicialización | Capas entrenadas | Máximo de épocas | LR |
|---|---|---|---|---|
| `imagenet_cabeza` | ImageNet | Cabeza nueva | 10 | 1e-3 |
| `imagenet_finetuning` | Mejor cabeza anterior | Cabeza + bloque conv5, sin BatchNorm | 10 adicionales | 1e-5 |
| `aleatorio` | Pesos aleatorios | Toda la red, incluida BatchNorm | 20 | 1e-4 |

Misma entrada 256×256, cabeza GAP → Dropout(0.3) → Dense(1, sigmoid), aumentos y split.
La conexión residual de ResNet suma la entrada del bloque a su transformación y facilita entrenar redes profundas.
Los presupuestos son iniciales; no demuestran convergencia ni equivalencia de costo entre estrategias.
Se selecciona el checkpoint con menor `val_loss`, con paciencia 5. **Val no es un test independiente**.
Umbral fijo 0.5; precision/recall/F1 binarios para la clase positiva, además de F1 macro y métricas por clase.

Referencias: [ResNet50 y preprocesamiento](https://keras.io/api/applications/resnet/resnet_models/),
[transfer learning y BatchNorm](https://keras.io/guides/transfer_learning/).

In [ ]:
# --- Setup: correr siempre primero ---
import os
import subprocess
import sys
from pathlib import Path

INTEGRANTE = os.environ.get("PDI_INTEGRANTE", "nombre")  # reemplazar por tu nombre

if "google.colab" in sys.modules:
    repo = Path("/content/PDI-TP")
    if repo.is_dir():
        subprocess.run(["git", "-C", str(repo), "pull", "--ff-only", "-q"], check=True)
    else:
        subprocess.run(["git", "clone", "-q", "https://github.com/candeperles/PDI-TP.git", str(repo)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(repo / "requirements-training.txt")], check=True)
else:
    # Funciona al abrir Jupyter desde la raíz o desde notebooks/.
    repo = next((p for p in (Path.cwd(), Path.cwd().parent)
                 if (p / "pdi_tp" / "entorno.py").is_file()), None)
    if repo is None:
        raise FileNotFoundError("Abrí Jupyter desde la raíz del repositorio o desde notebooks/.")
os.chdir(repo)
sys.path.insert(0, str(repo))

from pdi_tp import entorno, datos, resultados, split
rutas = entorno.preparar(INTEGRANTE)

## Configuración y entorno

Completar `INTEGRANTE` arriba. El setup de Colab instala dependencias de entrenamiento en lugar de
las de exploración. Si pide reinicio, reiniciar el runtime y volver a ejecutar.
La primera carga de ImageNet requiere Internet. En local instalar `requirements-training.txt`
en un entorno separado con Python compatible con TensorFlow; el entorno de exploración Python 3.14
no está verificado para entrenamiento. No se publican cambios ni se modifica el split al ejecutar.

In [ ]:
import json
import platform
from datetime import datetime
from importlib.metadata import version
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from pdi_tp import clasificacion as clf, resnet50 as rn

SEMILLA = 42
LADO = 256
BATCH = 32  # bajar a 16 si falta memoria en T4
EPOCAS_CABEZA = 10
EPOCAS_FINE = 10
EPOCAS_ALEATORIO = 20
UMBRAL = 0.5
AUGMENTATION = True
keras.utils.set_random_seed(SEMILLA)
tf.config.experimental.enable_op_determinism()
gpus = tf.config.list_physical_devices("GPU")
print("TensorFlow:", tf.__version__, "GPU:", gpus)
if not gpus:
    print("AVISO: sin GPU; los tres entrenamientos pueden tardar mucho. En Colab seleccionar T4.")
RUN = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
salida = rutas.resultados / f"resnet50_{RUN}"
salida.mkdir(parents=True, exist_ok=False)

## Referencia y preparación reproducible

Se auditan los 900 pares. Solo se permite excluir Z01/línea 1, de ancho cero, según criterios v1.
Se guardan un manifiesto binario y metadatos por caja, con clase y confianza originales.
No se necesita copiar imágenes para clasificación. Las cajas positivas pequeñas y las clases 1/2 se conservan.

**N06 y N16 siguen pendientes**: `y` permanece vacío y `y_original=0` conserva lo entregado.
Se guardan predicciones para las 162 imágenes. Las métricas completas y la selección de checkpoints
usan provisionalmente la referencia original; se añade análisis de sensibilidad sobre los 160 casos
sin duda registrada. Esto no confirma semánticamente las otras etiquetas ni reemplaza la segunda revisión.

In [ ]:
tabla, cajas = clf.preparar_tabla(rutas)
tabla.to_csv(salida / "manifiesto.csv", index=False)
cajas.to_csv(salida / "cajas_derivadas.csv", index=False)
train = tabla.query("split == 'train'").reset_index(drop=True)
val = tabla.query("split == 'val'").reset_index(drop=True)
PESOS = clf.pesos_clase(train.y.to_numpy(dtype=int))  # calculados únicamente con train
HUELLA = clf.huella_tabla(tabla)
display(tabla.groupby(["split", "y_original", "pendiente_revision"]).size().rename("imagenes").to_frame())
display(val.loc[val.pendiente_revision, ["archivo", "y", "y_original"]])
print("Pesos de clase:", PESOS, "\nHuella:", HUELLA)
config = dict(modelo="ResNet50", lado=LADO, batch=BATCH, semilla=SEMILLA,
    augmentation=AUGMENTATION, aumentos="flip horizontal/vertical + rotacion 0.25 vueltas",
    dropout=0.3, cabeza="GlobalAveragePooling2D, Dropout, Dense sigmoid",
    umbral=UMBRAL, clase_positiva=1, promedio="binary", pesos_clase=PESOS,
    criterio_datos="v1; excluir solo Z01 linea 1; conservar cajas pequenas y confianza",
    huella_manifiesto=HUELLA, n_train=len(train), n_val=len(val),
    evaluacion="provisional; referencia original; N06/N16 pendientes",
    monitor="val_loss", patience=5, optimizador="Adam", loss="binary_crossentropy",
    preprocesamiento="ResNet preprocess_input sobre RGB 0..255, incluido en modelo",
    epocas_cabeza=EPOCAS_CABEZA, epocas_fine=EPOCAS_FINE, epocas_aleatorio=EPOCAS_ALEATORIO)
clf.guardar_json(salida / "config.json", config)
clf.guardar_json(salida / "entorno.json", dict(python=platform.python_version(),
    paquetes={p: version(p) for p in ["tensorflow", "keras", "numpy", "pandas", "Pillow", "scikit-learn"]},
    gpu=[str(g) for g in gpus], detalle_gpu=[tf.config.experimental.get_device_details(g) for g in gpus]))

## Evaluación común y baseline

La matriz usa filas = referencia, columnas = predicción, orden `[sin paneles, con paneles]`.
Se exporta una fila por imagen con nombre, fuente, grupo, referencia, probabilidad y predicción
para la futura comparación con los detectores. Se informa también el baseline «siempre con paneles».
Los archivos `.keras` incluyen preprocesamiento y esperan imágenes RGB en 0..255.

In [ ]:
resumen = []

def evaluar_guardar(nombre, probabilidades, params, modelo=None):
    carpeta = salida / nombre
    carpeta.mkdir(exist_ok=True)
    p = np.asarray(probabilidades).reshape(-1)
    metricas = clf.evaluar(val.y_original.to_numpy(dtype=int), p, UMBRAL)
    mascara = ~val.pendiente_revision.to_numpy()
    metricas["sin_pendientes"] = clf.evaluar(val.loc[mascara, "y_original"].to_numpy(dtype=int), p[mascara], UMBRAL)
    pred = val[["archivo", "fuente", "grupo", "y", "y_original", "pendiente_revision"]].copy()
    pred["probabilidad"] = p
    pred["prediccion"] = (p >= UMBRAL).astype(int)
    pred.to_csv(carpeta / "predicciones_val.csv", index=False)
    por_fuente = {f: clf.evaluar(g.y_original.to_numpy(dtype=int), g.probabilidad, UMBRAL)
                  for f, g in pred.groupby("fuente")}
    metricas["por_fuente"] = por_fuente
    clf.guardar_json(carpeta / "metricas.json", metricas)
    parametros = dict(config, **params, run=RUN, carpeta=str(carpeta))
    resultados.guardar_experimento(rutas, f"resnet50_{RUN}_{nombre}", parametros, metricas)
    resumen.append(dict(experimento=nombre, **{k: metricas[k] for k in
                   ["accuracy", "precision", "recall", "f1", "f1_macro"]},
                   f1_sin_pendientes=metricas["sin_pendientes"]["f1"]))
    pd.DataFrame(resumen).to_csv(salida / "comparacion.csv", index=False)
    from sklearn.metrics import ConfusionMatrixDisplay
    fig, ax = plt.subplots(figsize=(5, 4))
    ConfusionMatrixDisplay(np.array(metricas["matriz_confusion"]),
        display_labels=["sin paneles", "con paneles"]).plot(ax=ax, colorbar=False)
    ax.set_title(f"{nombre} · referencia provisional")
    fig.tight_layout()
    fig.savefig(carpeta / "matriz_confusion.png", dpi=130)
    plt.show()
    plt.close(fig)
    if modelo is not None:
        clf.guardar_json(carpeta / "parametros_modelo.json", dict(total=modelo.count_params(),
            entrenables=int(sum(np.prod(w.shape) for w in modelo.trainable_weights))))
    return pred

def curvas(nombre):
    historia = pd.read_csv(salida / nombre / "historia.csv")
    fig, axs = plt.subplots(1, 2, figsize=(10, 3))
    for ax, metrica in zip(axs, ["loss", "accuracy"]):
        ax.plot(historia.epoch + 1, historia[metrica], label="train")
        ax.plot(historia.epoch + 1, historia["val_" + metrica], label="val")
        ax.set(xlabel="Época", ylabel=metrica, title=nombre)
        ax.legend()
    fig.tight_layout()
    fig.savefig(salida / nombre / "curvas.png", dpi=130)
    plt.show()
    plt.close(fig)

evaluar_guardar("baseline_siempre_panel", np.ones(len(val)), dict(estrategia="baseline"))

## 1. ImageNet: entrenar solo la cabeza

El extractor queda congelado. `preprocess_input` convierte RGB a BGR y resta las medias de ImageNet;
no dividir los píxeles por 255. Los aumentos solo se activan al entrenar.

In [ ]:
keras.backend.clear_session()
keras.utils.set_random_seed(SEMILLA)
train_ds = rn.dataset(train, LADO, BATCH, entrenamiento=True, semilla=SEMILLA)
val_ds = rn.dataset(val, LADO, BATCH)
modelo, base = rn.construir(LADO, preentrenado=True, augmentation=AUGMENTATION, semilla=SEMILLA)
modelo.summary()
rn.entrenar(modelo, train_ds, val_ds, salida / "imagenet_cabeza", EPOCAS_CABEZA, 1e-3, PESOS)
evaluar_guardar("imagenet_cabeza", modelo.predict(val_ds, verbose=0),
    dict(estrategia="cabeza", inicializacion="imagenet", lr=1e-3, epocas_max=EPOCAS_CABEZA), modelo)
curvas("imagenet_cabeza")

## 2. ImageNet: ajuste fino del bloque conv5

Continúa desde el mejor checkpoint de la cabeza, que ya fue restaurado. Se descongela conv5
excepto BatchNorm, se recompila con Adam nuevo y LR 1e-5. La base se llama con `training=False`.
Se registra la dependencia del experimento anterior y las épocas adicionales.

In [ ]:
rn.descongelar(base)
rn.entrenar(modelo, train_ds, val_ds, salida / "imagenet_finetuning", EPOCAS_FINE, 1e-5, PESOS)
evaluar_guardar("imagenet_finetuning", modelo.predict(val_ds, verbose=0),
    dict(estrategia="conv5_sin_batchnorm", inicializacion="mejor_imagenet_cabeza",
         checkpoint_inicial=str(salida / "imagenet_cabeza" / "mejor.keras"),
         lr=1e-5, epocas_adicionales_max=EPOCAS_FINE), modelo)
curvas("imagenet_finetuning")
del modelo, base, train_ds, val_ds

## 3. ResNet50 desde cero

Se crea una red nueva con `weights=None`, sin reutilizar pesos anteriores. Toda la red se entrena;
BatchNorm actualiza sus estadísticas. Se mantienen cabeza, aumentos y preprocesamiento para comparar.
Con 738 imágenes puede sobreajustar y necesitar mayor presupuesto; interpretar también las curvas.

In [ ]:
keras.backend.clear_session()
keras.utils.set_random_seed(SEMILLA)
train_ds = rn.dataset(train, LADO, BATCH, entrenamiento=True, semilla=SEMILLA)
val_ds = rn.dataset(val, LADO, BATCH)
modelo, base = rn.construir(LADO, preentrenado=False, augmentation=AUGMENTATION, semilla=SEMILLA)
rn.entrenar(modelo, train_ds, val_ds, salida / "aleatorio", EPOCAS_ALEATORIO, 1e-4, PESOS)
predicciones = evaluar_guardar("aleatorio", modelo.predict(val_ds, verbose=0),
    dict(estrategia="toda_la_red", inicializacion="aleatoria", lr=1e-4, epocas_max=EPOCAS_ALEATORIO), modelo)
curvas("aleatorio")

## Comparar e interpretar los resultados reales

La tabla se completa únicamente al ejecutar los entrenamientos. No hay métricas simuladas.
Comparar F1 por clase y F1 macro con el baseline, no solo accuracy. Revisar si los errores
se concentran por fuente, en cajas pequeñas o en anotaciones inciertas. La mejora con ajuste fino
debe acompañarse de curvas y presupuesto total; una corrida con semilla fija no mide variabilidad.
Falta repetir con otra arquitectura para completar el requisito global del ítem 2.

In [ ]:
display(pd.DataFrame(resumen).set_index("experimento"))
print("Resultados y checkpoints:", salida)
# Todas las predicciones mantienen los nombres para comparar con YOLO sobre el mismo val.
display(predicciones.query("prediccion != y_original").head(20))

## Recuperar un modelo guardado

`mejor.keras` conserva el modelo seleccionado; `ultimo.keras` guarda modelo y optimizador de la última
época completada. `historia.csv` registra el avance. Los checkpoints de cada experimento van a Drive
en Colab; una corrida nueva crea otra carpeta. Para recuperar, usar `keras.models.load_model(ruta)`.
No ejecutar nuevamente la celda de entrenamiento con la misma carpeta: se rechaza para no sobrescribir.